In [2]:
import torch
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89], # Token 1
        [0.55, 0.87, 0.66], # Token 2
        [0.57, 0.85, 0.64], # Token 4
        [0.22, 0.58, 0.33], # Token 3
        [0.77, 0.25, 0.10], # Token 5
        [0.05, 0.80, 0.55]  # Token 6

    ]
)

In [4]:
input_query = inputs[1]
input_query

tensor([0.5500, 0.8700, 0.6600])

In [5]:
torch.dot(input_query, inputs[0])

tensor(0.9544)

In [6]:
attn_score = torch.zeros(inputs.shape[0])
for i in range(inputs.shape[0]):
    attn_score[i] = torch.dot(input_query, inputs[i])
print (attn_score)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [7]:
sum_attn_score = torch.sum(attn_score)
normalized_attn_score = attn_score / sum_attn_score
normalized_attn_score

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])

In [ ]:
# what this do is normalize the attention scores by applying the softmax function because it ensures that the attention scores are positive and sum to 1, which is a requirement for attention mechanisms in neural networks. The softmax function transforms the raw attention scores into a probability distribution, allowing the model to focus on different parts of the input sequence based on their relevance to the current query.
normalized_attn_score = torch.softmax(attn_score, dim=0) 
normalized_attn_score

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [12]:
context_vector = torch.zeros(inputs.shape[1])
for i,i_x in enumerate(inputs):
    context_vector += normalized_attn_score[i] * i_x
context_vector

tensor([0.4419, 0.6515, 0.5683])

In [16]:
# Attn weights 

attn_scores = inputs @ inputs.T
attn_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [17]:
attn_scores = torch.softmax(attn_scores, dim = 1)
attn_scores

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [20]:
context_vector_2 = torch.empty(inputs.shape[1])
print(inputs.shape)
context_vector_2 = attn_scores @ inputs

torch.Size([6, 3])


In [21]:
context_vector_2


tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

In [ ]:
# A vector of the same dimensionality as the token embeddings, representing a weighted combination of the information from the tokens that x is attending to.
# its just a learned representation affected by what words surrounds it 

In [56]:
import torch

class SelfAttention(torch.nn.Module):
    def __init__(self, d_in, d_out, context_length,dropout, qkv_bias):
        super().__init__()
        self.Wq = torch.nn.Linear(d_in,d_out, bias = qkv_bias) # 2x4
        self.Wk = torch.nn.Linear(d_in,d_out, bias = qkv_bias)
        self.Wv = torch.nn.Linear(d_in,d_out, bias = qkv_bias)
        self.dropout = torch.nn.Dropout(dropout) 
        self.register_buffer(
        'mask',
        torch.triu(torch.ones(context_length, context_length),
        diagonal=1)
        )

        print(self.Wq.weight)
        print(self.Wq.weight.shape)
        print(d_in," , ",d_out)
    def forward(self, x):
        # Matrix Multiplication Wq @ Q ...
        Q = self.Wq(x) # (6,4).(4,4)
        K = self.Wk(x)
        V = self.Wv(x)
        # Calculating attn_scores
        attn_scores = torch.matmul(Q, K.T)
        context_length = attn_scores.shape[0]
        mask_simple = torch.triu(torch.ones(context_length, context_length),diagonal=1)
        attn_weights = attn_scores.masked_fill(mask_simple.bool(),-torch.inf)
        print (attn_weights)
        attn_scores = torch.softmax(attn_weights/K.shape[-1]**0.5,dim=1)
        print (attn_scores)
        context_vector =attn_scores @ V

        return context_vector


In [57]:
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89, 0.56], # Token 1
        [0.55, 0.87, 0.66, 0.6],    # Token 2  #6,4 * 4,2 = 6,2
        [0.57, 0.85, 0.64, 0.51], # Token 4
        [0.22, 0.58, 0.33, 0.12], # Token 3
        [0.77, 0.25, 0.10, 0.14], # Token 5
        [0.05, 0.80, 0.55, 0.02]  # Token 6

    ]
)

sa_x = SelfAttention(4,2,context_length=4,dropout=0.5,qkv_bias=True)
print (sa_x(inputs))

Parameter containing:
tensor([[ 0.0557, -0.3067, -0.0478,  0.1405],
        [-0.4600, -0.4527,  0.0377,  0.3286]], requires_grad=True)
torch.Size([2, 4])
4  ,  2
tensor([[-0.0607,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.0235, -0.1059,    -inf,    -inf,    -inf,    -inf],
        [-0.0154, -0.0984, -0.1299,    -inf,    -inf,    -inf],
        [-0.0523, -0.1324, -0.1581, -0.2625,    -inf,    -inf],
        [ 0.0463,  0.0023, -0.0215, -0.1753, -0.0689,    -inf],
        [-0.0817, -0.1847, -0.2160, -0.3317, -0.2558, -0.4111]],
       grad_fn=<MaskedFillBackward0>)
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5146, 0.4854, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3490, 0.3291, 0.3219, 0.0000, 0.0000, 0.0000],
        [0.2678, 0.2530, 0.2485, 0.2308, 0.0000, 0.0000],
        [0.2128, 0.2063, 0.2028, 0.1819, 0.1961, 0.0000],
        [0.1868, 0.1737, 0.1699, 0.1565, 0.1652, 0.1480]],
       grad_fn=<SoftmaxBackward0>)
tensor([[-0.1665, -0.6841],
   

In [47]:
torch.sum(torch.tensor([0.2003, 0.2011, 0.2009, 0.1996, 0.1981, 0.0000]))

tensor(1.)